# 🏰 Holdfast — Full-Duplex-Bench Reproduction

This notebook runs the Holdfast voice agent project on Google Colab.

**Requirements:**
- GPU runtime (T4 is fine) — go to `Runtime > Change runtime type > T4 GPU`
- API keys: LiveKit Cloud + OpenAI (only needed for the full benchmark)

**Sections:**
1. Upload & Setup
2. Install Dependencies
3. Offline Tests (free, no API keys)
4. Full Benchmark (needs API keys)
5. Download Results

---
## 1️⃣ Upload the Project

**Option A:** Upload a zip of the `holdfast` folder (recommended if no GitHub repo).

**Option B:** Clone from GitHub (uncomment and edit the git clone line).

In [ ]:
import os
import glob
import shutil
import zipfile

# 1. Direct path setup using a raw string (r"...") to prevent Windows backslash errors
target_zip = r"C:\Users\Himanish Srivastava\Downloads\holdfast\holdfast.zip"

# Double check if Windows can actually see the file at this exact location
if not os.path.exists(target_zip):
    raise FileNotFoundError(f"Python cannot find the file at: {target_zip}\n"
                            f"Please verify the folder names or if the file extension is exactly '.zip'.")

print(f"Success! Found file. Extracting {target_zip}...")

# 2. Set up extraction directory inside your current Jupyter folder
extract_dir = os.path.abspath('./holdfast_project')

# Clean up any existing directory to avoid mixing files on re-runs
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)

# 3. Extract the ZIP file safely
with zipfile.ZipFile(target_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# 4. Auto-detect the project root (handles nested folders)
candidates = glob.glob(os.path.join(extract_dir, '**/reproduce.sh'), recursive=True)
if candidates:
    PROJECT_ROOT = os.path.dirname(candidates[0])
else:
    raise FileNotFoundError('Could not find reproduce.sh inside the extracted files.')

print(f'\n✅ Project root found: {PROJECT_ROOT}')
os.chdir(PROJECT_ROOT)

# 5. List the contents of the project root
print("Current directory contents:")
print(os.listdir('.'))


In [ ]:
# === OPTION B: Clone from GitHub (uncomment and edit) ===
# !git clone https://github.com/YOUR_USERNAME/holdfast.git /content/holdfast
# %cd /content/holdfast
# PROJECT_ROOT = '/content/holdfast'

---
## 2️⃣ Preflight Checks & Install Dependencies

In [ ]:
# Check prerequisites (Windows-compatible syntax)
print("Python:")
!python --version

print("\nFFmpeg:")
# Stripped out 'head -1' and Linux pipes to prevent Windows errors
!ffmpeg -version

print("\nGit:")
!git --version

print("\nGPU:")
# Uses '||' alternative or standalone check safely
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>nul || echo No GPU detected — switch to GPU runtime!


In [ ]:
import os
import shutil

# 1. Map out paths to your specific environment
venv_path = os.path.abspath("./.venv-agent")
venv_python = os.path.join(venv_path, "Scripts", "python.exe")

# 2. Bootstrap pip directly into this environment
print("Bootstrapping pip into the virtual environment...")
!"{venv_python}" -m ensurepip --upgrade

# 3. Upgrade pip to its latest version
print("\nUpgrading pip...")
!"{venv_python}" -m pip install --upgrade pip

# 4. Install the required tools
print("\nInstalling requirements-agent.txt...")
!"{venv_python}" -m pip install -r requirements-agent.txt

# 5. Verify the package list
print("\n✅ Success! Verification of installed packages:")
!"{venv_python}" -m pip list



In [ ]:
import os
print("Folders in your current directory:")
print([f for f in os.listdir('.') if os.path.isdir(f)])


---
## 3️⃣ Offline Tests (No API Keys Needed)

These verify the control layer logic without making any API calls.

In [ ]:
import subprocess
import os

# Target the correct Windows folder layout
venv_python = os.path.abspath(r".\.venv-agent\Scripts\python.exe")

print("🚀 Starting 93 unit tests via pytest...")

# Run pytest safely and output results directly to the screen
subprocess.run([venv_python, "-m", "pytest", "-q", "tests"])


In [ ]:
import os
import sys
import subprocess

# 1. Emulate 'PYTHONPATH=scripts:.' by adding them to the environment dictionary
env = os.environ.copy()
env["PYTHONPATH"] = f"scripts{os.pathsep}."

# 2. Point to the proper Windows Python executable
venv_python = os.path.abspath(r".\.venv-agent\Scripts\python.exe")
harness_script = r"scripts\harness_selftest.py"

print("🚀 Launching harness self-test...")

# 3. Run the script safely
subprocess.run([venv_python, harness_script], env=env)


In [ ]:
import subprocess
import os

# Target the correct Windows executable folder layout
venv_python = os.path.abspath(r".\.venv-agent\Scripts\python.exe")
sim_script = r"scripts\drive_sim.py"

print("🚀 Starting the Drive extension simulation...")

# Run the simulation script safely
subprocess.run([venv_python, sim_script])


---
## 4️⃣ Full Benchmark (Needs API Keys)

⚠️ **This costs real money** — it uses the OpenAI Realtime API for the agent and gpt-4o for the judge.

You need:
- **LiveKit Cloud** — free project at https://cloud.livekit.io
- **OpenAI API key** — with access to the Realtime API

### Set your API keys
Use Colab Secrets (🔑 in sidebar) for security, or paste directly below.

In [ ]:
import os
from getpass import getpass

keys = ["LIVEKIT_URL", "LIVEKIT_API_KEY", "LIVEKIT_API_SECRET", "OPENAI_API_KEY"]
values = {k: os.environ.get(k) or getpass(f"{k}: ") for k in keys}

with open(".env", "w", encoding="utf-8") as f:      # .env is in .gitignore
    for k, v in values.items():
        f.write(f"{k}={v}\n")

missing = [k for k, v in values.items() if not v]
print("Missing:", missing if missing else "none. .env written (never commit this file)")

In [ ]:
import os
import subprocess
from dotenv import dotenv_values

FDB_REPO = "https://github.com"
FDB_COMMIT = "3e799c45a045256f47d5f1c9cda90157e2d2ec9e"
fdb_dest = os.path.abspath(r"./third_party/Full-Duplex-Bench")
v3_dir = os.path.join(fdb_dest, "v3")

# 1. Clone and sync pinned commit
if not os.path.exists(fdb_dest):
    print("Cloning Full-Duplex-Bench repository...")
    subprocess.run(["git", "clone", FDB_REPO, fdb_dest])

print(f"Syncing code to commit {FDB_COMMIT[:7]}...")
subprocess.run(["git", "-C", fdb_dest, "fetch", "origin", FDB_COMMIT])
subprocess.run(["git", "-C", fdb_dest, "checkout", FDB_COMMIT])

# 2. Write credentials from your root .env to the v3 submodule location
root_env = os.path.abspath("./.env")
if os.path.exists(root_env):
    config = dotenv_values(root_env)
    env_local = os.path.join(v3_dir, ".env.local")
    with open(env_local, "w", encoding="utf-8") as f:
        for key in ["LIVEKIT_URL", "LIVEKIT_API_KEY", "LIVEKIT_API_SECRET", "OPENAI_API_KEY"]:
            f.write(f"{key}={config.get(key, '')}\n")
    print("✅ Submodule credentials synchronized.")
else:
    print("❌ Root .env file missing. Run your configuration cell again first.")


In [ ]:
import os
import shutil
import urllib.request
import subprocess

# 1. Clean up old environment using cross-platform shutil instead of !rm
venv_path = os.path.abspath("./.venv-harness")
if os.path.exists(venv_path):
    print("Clearing out old .venv-harness environment...")
    shutil.rmtree(venv_path)

# 2. Create the virtual environment natively
print("Creating virtual environment without pip...")
subprocess.run(["python", "-m", "venv", "--without-pip", ".venv-harness"])

# 3. Download the pip installer bootstrapper
print("Downloading get-pip.py...")
urllib.request.urlretrieve("https://bootstrap.pypa.io/get-pip.py", "get-pip.py")

# 4. Correctly reference the Windows Python executable inside the new venv-harness
venv_python = os.path.join(venv_path, "Scripts", "python.exe")

# 5. Manually install pip using the Windows path layout
print("Manually installing pip inside the virtual environment...")
subprocess.run([venv_python, "get-pip.py", "-q"])

# 6. Install the harness dependencies file
requirements_file = "requirements-harness.txt"
if os.path.exists(requirements_file):
    print(f"Installing {requirements_file}...")
    subprocess.run([venv_python, "-m", "pip", "install", "-q", "-r", requirements_file])
else:
    print(f"⚠️ {requirements_file} not found in this folder.")

# 7. Clean up the installer script safely
if os.path.exists("get-pip.py"):
    os.remove("get-pip.py")

print('\n✅ Harness environment installed successfully with pip!')

# 8. Verification step - list your packages to be 100% sure it worked
print("\nVerifying installed harness packages:")
subprocess.run([venv_python, "-m", "pip", "list"])


In [ ]:
import os

if os.path.exists("reproduce.sh"):
    with open("reproduce.sh", "r", encoding="utf-8") as f:
        print("--- Contents of reproduce.sh ---")
        print(f.read())
else:
    print("❌ reproduce.sh not found in the current directory.")


In [ ]:
import re

# Patch reproduce.sh to handle Colab's broken ensurepip during virtual environment creation
with open('reproduce.sh', 'r') as f:
    script_content = f.read()

# Replace standard venv creation with our robust pip-bootstrapping method
old_setup_pattern = r"python3\s+-m\s+venv\s+\.(venv-agent|venv-harness)"

patch_code = """python3 -m venv --without-pip .\\1 && curl -sS https://bootstrap.pypa.io/get-pip.py | ./.\\1/bin/python"""

patched_content = re.sub(old_setup_pattern, patch_code, script_content)

with open('reproduce.sh', 'w') as f:
    f.write(patched_content)

print("✅ Successfully patched reproduce.sh to bypass the broken ensurepip block!")

---
## 5️⃣ View & Download Results

In [ ]:
# Show the summary
import glob
summaries = sorted(glob.glob('results/runs/*/summary.md'))
if summaries:
    latest = summaries[-1]
    print(f'📄 Latest results: {latest}\n')
    with open(latest) as f:
        print(f.read())
else:
    print('No results found yet — run the benchmark first (Section 4)')

In [ ]:
import os
import shutil

# Target your local results directory
source_dir = os.path.abspath("./results")
output_zip = os.path.abspath("./holdfast_results")

if os.path.exists(source_dir) and os.listdir(source_dir):
    print(f"Compressing results folder from: {source_dir}...")
    # This natively zips the folder on Windows without any external tools
    shutil.make_archive(output_zip, 'zip', source_dir)
    print(f"\n✅ Success! Your ZIP file has been created directly on your PC at:\n{output_zip}.zip")
else:
    print("⚠️ The 'results' directory is currently empty or doesn't exist yet.")
    print("Please make sure your inference benchmarks (Step 4) finished running first!")


In [1]:
import os, subprocess, pathlib

PROJECT_ROOT = r"C:\Users\Himanish Srivastava\Downloads\holdfast\holdfast"   # <- check this
os.chdir(PROJECT_ROOT)

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr)

# keep secrets and heavy folders out of the repo
gi = pathlib.Path(".gitignore")
have = gi.read_text() if gi.exists() else ""
need = [".env", ".venv-*/", "third_party/", "results/runs/",
        "holdfast_project/", ".ipynb_checkpoints/", "__pycache__/", ".pytest_cache/"]
gi.write_text(have + "".join(n + "\n" for n in need if n not in have))

git("init", "-b", "main")
git("config", "user.name", "Himanish Srivastava")      # <- your name
git("config", "user.email", "you@example.com")         # <- the email on your GitHub account
git("add", ".")

files = subprocess.run(["git", "ls-files"], capture_output=True, text=True).stdout.splitlines()
print("files to upload:", len(files))
print("over 50 MB:", [f for f in files if os.path.exists(f) and os.path.getsize(f) > 50e6])
print(".env tracked:", [f for f in files if f.endswith(".env")])

Reinitialized existing Git repository in C:/Users/Himanish Srivastava/Downloads/holdfast/holdfast/.git/




files to upload: 50
over 50 MB: []
.env tracked: []


In [2]:
git("commit", "-m", "Holdfast: interruptible real-time voice agent + notebook")
git("remote", "add", "origin", "https://github.com/himanish-srivastava/Realtime-Interruptable-AI-Agents.git")
git("push", "-u", "origin", "main")

[main 7575f9f] Holdfast: interruptible real-time voice agent + notebook
 1 file changed, 59 insertions(+), 40 deletions(-)

error: remote origin already exists.

error: RPC failed; curl 56 Recv failure: Connection was reset
send-pack: unexpected disconnect while reading sideband packet
fatal: the remote end hung up unexpectedly
Everything up-to-date



In [3]:
import os, re, shutil, stat, subprocess, pathlib

PROJECT_ROOT = r"C:\Users\Himanish Srivastava\Downloads\holdfast\holdfast"   # same folder as before
REMOTE = "https://github.com/himanish-srivastava/Realtime-Interruptable-AI-Agents.git"
NAME, EMAIL = "Himanish Srivastava", "himanishsri624@gmail.com"                        # <- email on your GitHub account
os.chdir(PROJECT_ROOT)

def git(*args, show=True):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if show:
        print(r.stdout + r.stderr)
    return r

def _force(func, path, exc):            # git marks its files read-only on Windows
    os.chmod(path, stat.S_IWRITE); func(path)

# 1. discard the old history (it contains the commit with the key)
if os.path.exists(".git"):
    shutil.rmtree(".git", onerror=_force)

# 2. keep secrets and heavy folders out
gi = pathlib.Path(".gitignore")
have = gi.read_text() if gi.exists() else ""
need = [".env", ".venv-*/", "third_party/", "results/runs/", "holdfast_project/",
        ".ipynb_checkpoints/", "__pycache__/", ".pytest_cache/"]
gi.write_text(have + "".join(n + "\n" for n in need if n not in have))

# 3. fresh repository
git("init", "-b", "main"); git("config", "user.name", NAME); git("config", "user.email", EMAIL)
git("add", ".")

# 4. safety checks on exactly what would be uploaded
files = git("ls-files", show=False).stdout.splitlines()
big   = [f for f in files if os.path.exists(f) and os.path.getsize(f) > 50e6]
envs  = [f for f in files if os.path.basename(f) == ".env"]
rx    = re.compile(r"sk-[A-Za-z0-9_\-]{20,}|LIVEKIT_API_SECRET\s*[=:]\s*['\"]?[A-Za-z0-9]{8,}"
                   r"|wss://[a-z0-9\-]+\.livekit\.cloud")
leaks = []
for f in files:
    if f.lower().endswith((".png", ".jpg", ".wav", ".zip", ".pyc", ".pdf")):
        continue
    try:
        if rx.search(open(f, encoding="utf-8", errors="ignore").read()):
            leaks.append(f)
    except OSError:
        pass
print(f"files: {len(files)} | over 50 MB: {big} | .env tracked: {envs} | secret-like: {leaks}")
if big or envs or leaks:
    raise SystemExit("STOPPED: fix the items above (edit the file, or add it to .gitignore and re-run).")

# 5. one clean commit, then push
git("commit", "-m", "Holdfast: interruptible real-time voice agent + notebook")
n = len(git("log", "--oneline", show=False).stdout.splitlines())
print("commits in history:", n)
assert n == 1, "history should be exactly one commit"
git("remote", "remove", "origin", show=False)
git("remote", "add", "origin", REMOTE)
git("push", "-u", "origin", "main")

Initialized empty Git repository in C:/Users/Himanish Srivastava/Downloads/holdfast/holdfast/.git/




files: 50 | over 50 MB: [] | .env tracked: [] | secret-like: ['.env.example', 'holdfast_colab.ipynb']


SystemExit: STOPPED: fix the items above (edit the file, or add it to .gitignore and re-run).

C:\Users\Himanish Srivastava\AppData\Local\Programs\Python\Python310\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
